# Triple Barrier Method

This notebook will cover partial exercise answers:
* Exercise 3.1
* Exercise 3.2
* Exercise 3.3

As we go along, there will be some explanations. 

More importantly, this method can be applied not just within mean-reversion strategy but also other strategies as well. Most of the functions below can be found under research/Labels.

Contact: boyboi86@gmail.com

In [ ]:
import numpy as np
import pandas as pd
import cqrlib as rs
import matplotlib.pyplot as plt

%matplotlib inline

p = print

#pls take note of version
#numpy 1.17.3
#pandas 1.0.3
#sklearn 0.21.3

dollar = pd.read_csv('./sample-data/dollar_bars.csv', 
                 sep=',', 
                 header=0, 
                 parse_dates = True, 
                 index_col=['date_time'])

# pandas 3.0 parses datetimes as datetime64[us]; cqrlib requires datetime64[ns]
dollar.index = dollar.index.as_unit('ns')

In [ ]:
d_vol = rs.vol(dollar['close'], span0 = 50)
print(d_vol)
print(d_vol.mean())

In [ ]:
# d_vol is a return (~0.55%); cs_filter diffs are price points, so scale by price level
events = rs.cs_filter(dollar['close'], 
                    limit = d_vol.mean() * dollar['close'].mean())

print(events)
print(events + pd.Timedelta("1day"))

In [ ]:
vb = rs.vert_barrier(data = dollar['close'], 
                 events = events, 
                 period = 'days', 
                 freq = 1)

vb # Show some example output

In [ ]:
data_test = pd.DataFrame(index=events).assign(data=dollar["close"])
print("===========")
print(dollar["close"])
print("===========")
print(data_test)
print(type(data_test))
print("===========")
print(data_test.squeeze())
print(type(data_test.squeeze()))

In [ ]:
tb = rs.tri_barrier(data = dollar['close'], 
                events = events, 
                trgt = d_vol, 
                min_req = 0.002, 
                num_threads = 3, 
                ptSl = [1,1], 
                t1 = vb, 
                side = None)

tb # Show some example

# the pandas obj will break the data up process it then stich it back into 1 piece again. (See below)
# this will only happen when you use pandas obj multiprocess func using num_threads > 1.

# if you scroll all the way to the bottom, that is your final dataframe output.

In [ ]:
m_label = rs.meta_label(data = dollar['close'],
                      events = tb,
                      drop = False)

m_label # Show some example

# previously when we run tri_bar func, NaT is present. However once func is passed to labels, these NaTs will be dropped.
# There is an in-built drop func that will trigger the below drop_label func as well.
# change drop = False to float value i.e. 0.05

> AFML page 54 section 3.9
>
> "Some ML classifiers do not perform well when data samples are too imbalanced. 
>  In those circumstances, it is preferably to drop those rare labels and focus on more common outcomes."

In [ ]:
m_label['bin'].value_counts()

# Here is a quick look at our 'bin' values.
# Apparently we have a rare label, bin = 0

In [ ]:
m_label['bin'].value_counts(normalize = True)

# basically it's 0.003602 of all our metalabels. Max is 1

In [ ]:
drop_meta_label = rs.drop_label(events = m_label, 
                                min_pct = 0.05)

drop_meta_label # Show some example

# In the below case we dropped all bin = 0, while keeping only 1 & -1